### Packages & Settings

In [ ]:
import sys  
sys.path.insert(0, '')  

In [ ]:
import datetime
import numpy as np
import pandas as pd 
from functions.preprocessing import DataExploration  
from functions.train import XGBoostModel
import pyodbc  
import pandas as pd  
import openai
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb  
from sklearn.metrics import accuracy_score, f1_score, classification_report, fbeta_score, confusion_matrix 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  
import joblib

### Load dataset

In [ ]:
# Read the CSV file of the customer base
customers  = pd.read_csv('../data/CUSTOMERS.csv',sep=";") 

In [ ]:
# Read policy file 
 
# Set up the connection string  
conn_str = (  
    r'DRIVER={SQL Server};'  
    r'SERVER=InsuranceServer;'  
    r'DATABASE=InsuranceDB;'  
    r'Trusted_Connection=yes;'  
)  
  
# Create the connection  
conn = pyodbc.connect(conn_str)  
  
# Write the SQL query  
sql_query = 'SELECT * FROM Policies WHERE Status = "Active"'  
  
# Use pandas to execute the query and store the result in a DataFrame  
policies = pd.read_sql(sql_query, conn)  
  
# Don't forget to close the connection  
conn.close()  

### Data exploration

In [ ]:
# Get a feel of the data: klanten
customers.dtypes

# Select only the columns of interest  
customers = customers.filter(items=['customer_id', 'birthdate', 'satisfaction_score', 'satisfaction', 'complaints', 'satisfaction_openquestion', 'postalcode', 'sexe']) 
policies = policies.filter(items=['policynr', 'customer_id', 'productcode', 'startdate', 'enddate'])

In [ ]:
# print nans
DataExploration.display_nans(customers)  
DataExploration.display_nans(policies)  


In [ ]:
# replace NaN
customers=DataExploration.fill_nans(customers)

In [ ]:
# create age from birthdate 
customers['age'] = ((datetime.now() - customers['birthdate']).dt.days // 365)

In [ ]:
# startdatum and einddatum have NANs. 
# Probably einddatum means the policy is still active, 
# which we can use as information for churn/no churn
# For startdatum, we might need to fill the NANs or continue as it is (only) 773 out of 9301.
# Possibly this is happening when the start date is before 2020.

# Check startdatum where NAN
print(policies[policies['startdate'].isnull()])
type(policies['startdate'][0])

# Convert to datetime
policies['startdate'] = pd.to_datetime(policies['startdate'])
policies['enddate'] = pd.to_datetime(policies['enddate'])

# print min startdate
print(policies['startdate'].min()) # = 1948, so assumption is incorrect. Missing NAN cannot be explained

# Drop rows where 'startdate' is NaN  
policies = policies.dropna(subset=['startdate'])  

In [ ]:
# add sociodemo data from cbs 
geodata=pd.read_csv('..data/exportCBSpc6_2014.csv')
geodata = geodata.filter(items=['PC6','inwoners','stedelijkheid','tweeouderhuishoudens','woningwaarde','perc_uitkeringen'])

In [ ]:
# convert open answers customer survey to satisfaction score
  
openai.api_key = ''  
  
def get_satisfaction_score(text):  
    response = openai.Completion.create(  
      engine="text-davinci-002",  
      prompt=f"This is a customer review: \"{text}\". Is the customer happy?",  
      temperature=0.5,  
      max_tokens=60  
    )  
  
    # Interpret the response  
    if 'yes' in response.choices[0].text.lower():  
        return 'satisfactory'  
    else:  
        return 'unsatisfactory'  
  
# Apply the function to the 'satisfaction_openquestions' column  
customers['satisfaction_score_2'] = customers['satisfaction_openquestions'].apply(get_satisfaction_score)    

In [ ]:
# merge datasets
df = pd.merge(customers, policies, on='customer_id', how='inner')
df=pd.merge(df,geodata,left_on='postalcode',right_on='PC6')


In [ ]:
df.head(5)

In [ ]:
df.to_csv("../data/trainset_20241001.csv",sep=";")

### Model training

In [ ]:
# target 
df['target'] = np.where(df['enddate'].notnull(), 1, 0)

In [ ]:
# keep interesting columns 
df.drop(columns=["customer_id", "policynr"], inplace=True)  
df=df[['satisfaction_score','satisfaction','complaints','productcode','satisfaction_score2','sexe','age','inwoners','stedelijkheid','tweeouderhuishoudens','woningwaarde','perc_uitkeringen']]

In [ ]:
# handle categorical features
  
cat_features = ['satisfaction', 'productcode','satisfaction_score2','sexe','stedelijkheid']  
   
le = LabelEncoder()  
ohe = OneHotEncoder()  
  
for col in cat_features:  
    # Label Encoding  
    df[col + '_le'] = le.fit_transform(df[col])  
  
    # One-Hot Encoding  
    cat_feature_ohe = ohe.fit_transform(df[col + '_le'].values.reshape(-1,1)).toarray()  
    dfOneHot = pd.DataFrame(cat_feature_ohe, columns = [col + "_" + str(int(i)) for i in range(cat_feature_ohe.shape[1])])  
    df = pd.concat([df, dfOneHot], axis=1)  
      
    # Drop the label encoded column  
    df.drop([col + '_le'], axis=1, inplace=True)  



In [ ]:
# train test
X = df.drop(columns=['target'])
y = df[['target']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y['target'])

In [ ]:
model = XGBoostModel()  
  
def objective(trial):  
    dtrain = xgb.DMatrix(X_train, label=y_train)  
  
    param = {  
        "silent": 1,  
        "objective": 'binary:logistic',  
        "eval_metric": 'logloss',  
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),  
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),  
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),  
    }  
  
    if param["booster"] == "gbtree" or param["booster"] == "dart":  
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)  
        param["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)  
        param["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)  
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])  
  
    bst = xgb.train(param, dtrain)  
    preds = bst.predict(xgb.DMatrix(X_test))  
    pred_labels = np.rint(preds)  
    accuracy = accuracy_score(y_test, pred_labels)  
    return accuracy  
  
model.train_and_validate_xgboost(X_train, X_test, y_train, y_test, objective)  




In [ ]:

# Save model to file  
joblib.dump(model, 'xgboost_model.pkl') 